In [2]:
import pandas as pd
import json
import re
from pathlib import Path
from sklearn.model_selection import train_test_split

# csv 파일 불러오기

In [2]:
import pandas as pd

# 정규화된 CSV 다시 불러오기
df_norm = pd.read_csv("C:/project_sep/csv/antibiotics_c_code_normalized.csv")

print("불러온 데이터 샘플:")
print(df_norm.head())

print("전체 행 개수:", len(df_norm))
print("고유한 C-Code 개수:", df_norm["C-Code"].nunique())

불러온 데이터 샘플:
                                 제품명    C-Code  \
0       유한짓정100mgYuhanzid Tab. 100mg  K-000040   
1                 후라시닐정Flasinyl Tab.  K-000066   
2  마이암부톨제피정400mgMyambutol Tab. 400mg  K-000080   
3      앰씰린캡슐250mgAmcillin Cap. 250mg  K-000105   
4                   셉트린정Septrin Tab.  K-000114   

                            제품명_norm  
0       유한짓정100mgyuhanzid tab. 100mg  
1                 후라시닐정flasinyl tab.  
2  마이암부톨제피정400mgmyambutol tab. 400mg  
3      앰씰린캡슐250mgamcillin cap. 250mg  
4                   셉트린정septrin tab.  
전체 행 개수: 343
고유한 C-Code 개수: 343


# 경로설정

In [6]:
# 데이터 경로
img_dir = Path(r"C:/project_sep/data/single_folder")  # 단일 이미지 폴더
json_dir = Path(r"C:/project_sep/data/flat_json")  # JSON 라벨 폴더(재귀 탐색)
csv_path = Path(
    r"C:/project_sep/csv/antibiotics_c_code_normalized.csv"
)  # 제품명↔C-Code CSV

print("이미지 폴더:", img_dir)
print("JSON 폴더:", json_dir)

이미지 폴더: C:\project_sep\data\single_folder
JSON 폴더: C:\project_sep\data\flat_json


In [4]:
df_csv = pd.read_csv(csv_path)

print("CSV 샘플:")
print(df_csv.head())

# 제품명과 C-Code 매핑 (딕셔너리)
name_to_ccode = {}
for i, row in df_csv.iterrows():
    name = str(row["제품명_norm"])
    code = str(row["C-Code"])
    name_to_ccode[name] = code

print("총 매핑 개수:", len(name_to_ccode))

CSV 샘플:
                                 제품명    C-Code  \
0       유한짓정100mgYuhanzid Tab. 100mg  K-000040   
1                 후라시닐정Flasinyl Tab.  K-000066   
2  마이암부톨제피정400mgMyambutol Tab. 400mg  K-000080   
3      앰씰린캡슐250mgAmcillin Cap. 250mg  K-000105   
4                   셉트린정Septrin Tab.  K-000114   

                            제품명_norm  
0       유한짓정100mgyuhanzid tab. 100mg  
1                 후라시닐정flasinyl tab.  
2  마이암부톨제피정400mgmyambutol tab. 400mg  
3      앰씰린캡슐250mgamcillin cap. 250mg  
4                   셉트린정septrin tab.  
총 매핑 개수: 343


# 이미지 + JSON 매핑하기

In [7]:
rows = []
matched, skipped_combo, bad_json, missing_img = 0, 0, 0, 0

total = len(list(json_dir.glob("*.json")))
print("총 JSON 파일 수:", total)

for j_idx, jpath in enumerate(json_dir.glob("*.json"), start=1):
    prefix = jpath.stem.split("_")[0]  # ex) "K-041374" or "K-016235-027733..."

    # 조합 JSON (K-xxxx-xxxx 형태) → 스킵
    if prefix.count("-") > 1:
        skipped_combo += 1
        continue

    k_code = prefix  # 단일 K-code

    # JSON 열기
    try:
        with open(jpath, "r", encoding="utf-8") as f:
            data = json.load(f)
    except:
        try:
            with open(jpath, "r", encoding="cp949", errors="ignore") as f:
                data = json.load(f)
        except:
            bad_json += 1
            continue

    if "images" not in data or len(data["images"]) == 0:
        bad_json += 1
        continue

    info = data["images"][0]
    file_name = info.get("file_name")
    product = info.get("dl_name")  # 제품명

    if not file_name:
        bad_json += 1
        continue

    img_path = img_dir / file_name
    if not img_path.exists():
        missing_img += 1
        continue

    rows.append([str(img_path), k_code, product])
    matched += 1

    # 진행률 로그: 1000개 단위 + 10% 단위
    if j_idx % 1000 == 0 or j_idx % (total // 10) == 0 or j_idx == total:
        print(
            f"[진행] {j_idx}/{total} "
            f"매칭 {matched}, 조합스킵 {skipped_combo}, "
            f"깨진JSON {bad_json}, PNG없음 {missing_img}"
        )

print("=== 최종 결과 ===")
print("총 JSON 파일 수:", total)
print("단일 매칭된 데이터 수:", matched)
print("조합 JSON 스킵 수:", skipped_combo)
print("깨진 JSON 수:", bad_json)
print("PNG 없는 경우:", missing_img)

총 JSON 파일 수: 160056
[진행] 1000/160056 매칭 1000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 2000/160056 매칭 2000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 3000/160056 매칭 3000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 4000/160056 매칭 4000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 5000/160056 매칭 5000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 6000/160056 매칭 6000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 7000/160056 매칭 7000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 8000/160056 매칭 8000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 9000/160056 매칭 9000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 10000/160056 매칭 10000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 11000/160056 매칭 11000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 12000/160056 매칭 12000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 13000/160056 매칭 13000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 14000/160056 매칭 14000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 15000/160056 매칭 15000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 16000/160056 매칭 16000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 16005/160056 매칭 16005, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 17000/160056 매칭 17000, 조합스킵 0, 깨진JSON 0, PNG없음 0
[진행] 18000/160056 매칭 18000

# DataFrame 으로 만들기

In [3]:
# DataFrame 생성
df = pd.DataFrame(rows, columns=["image_path", "k_code", "product_name"])

# K-code를 기준으로 class_id 숫자 라벨 생성
df["class_id"] = df["k_code"].astype("category").cat.codes

print("최종 이미지 수:", len(df))
print("클래스 수:", df["class_id"].nunique())
print(df.head())

NameError: name 'rows' is not defined

# 데이터 분할

In [9]:
from sklearn.model_selection import train_test_split

# 8:1:1

# train = 80%
train_df, temp_df = train_test_split(
    df, train_size=0.8, stratify=df["class_id"], random_state=42
)

# 남은 20% 중 val, test 절반씩
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, stratify=temp_df["class_id"], random_state=42
)

# split 컬럼 추가
train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

final_df = pd.concat([train_df, val_df, test_df], ignore_index=True)


print("Train 개수:", len(train_df))
print("Val 개수:", len(val_df))
print("Test 개수:", len(test_df))

Train 개수: 128044
Val 개수: 16006
Test 개수: 16006


# Samples.csv 저장

In [ ]:
SAVE_DIR = Path(r"C:/project_sep/csv")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

save_path = SAVE_DIR / "samples.csv"
final_df.to_csv(save_path, index=False, encoding="utf-8")

print("samples.csv 저장 완료:", save_path)
print(final_df.head())

samples.csv 저장 완료: C:\project_sep\csv\samples.csv
                                          image_path    k_code  \
0  C:\project_sep\data\single_folder\K-040599_0_2...  K-040599   
1  C:\project_sep\data\single_folder\K-044528_0_2...  K-044528   
2  C:\project_sep\data\single_folder\K-042278_0_0...  K-042278   
3  C:\project_sep\data\single_folder\K-042274_0_2...  K-042274   
4  C:\project_sep\data\single_folder\K-042573_0_2...  K-042573   

           product_name  class_id  split  
0  큐엔디1000연질캡슐 10mg/PTP        93  train  
1          로수암핀정 10/5mg       425  train  
2            징코에프정 80mg       254  train  
3   킥코프에프연질캡슐 200mg/PTP       252  train  
4          나스타제정 30mg/병       276  train  


# 클래스 매핑 확인

In [ ]:
import pandas as pd

# samples.csv 불러오기
df = pd.read_csv(r"C:/project_sep/csv/samples.csv")

# 고유 class_id, k_code, product_name 매핑
label_map = (
    df[["class_id", "k_code", "product_name"]]
    .drop_duplicates()
    .sort_values("class_id")
    .reset_index(drop=True)
)

# 보기 좋게 출력
for i, row in label_map.iterrows():
    print(f"{row['class_id']}: {row['k_code']}  ({row['product_name']})")

# 필요하다면 CSV로 저장
map_path = r"C:/project_sep/csv/label_map.csv"
label_map.to_csv(map_path, index=False, encoding="utf-8")
print("라벨 매핑 저장 완료:", map_path)

0: K-039148  (듀카브정60/5밀리그램)
1: K-039167  (하이소린정 2mg)
2: K-039168  (베아로탄플러스정)
3: K-039169  (베아로탄플러스프로정)
4: K-039170  (베아로탄플러스에프정)
5: K-039184  (넥스디핀정2.5mg)
6: K-039201  (셀트리온레보플록사신정 100mg)
7: K-039208  (트란시노2정 187.5mg/PTP)
8: K-039236  (엑세타민연질캡슐 20mg/포)
9: K-039248  (덴티골드캡슐 150mg/PTP)
10: K-039262  (울트란세미정)
11: K-039266  (셀룬정 200mg/병)
12: K-039272  (레바렉정)
13: K-039293  (젠빅플러스연질캡슐 500mg/PTP)
14: K-039306  (탐루신디서방정)
15: K-039320  (뉴암로디프정5밀리그램)
16: K-039325  (스카드비정)
17: K-039337  (크레진정 5mg)
18: K-039338  (크레진정 10mg)
19: K-039346  (비보존탐스로신서방정 0.2mg)
20: K-039362  (아사톱장용정 100mg)
21: K-039393  (로수바틴정 5mg)
22: K-039478  (크레로스정 10mg)
23: K-039494  (프릴린캡슐 75mg)
24: K-039504  (알푸론정)
25: K-039513  (넥리카캡슐 75mg)
26: K-039550  (디오브이정 80mg)
27: K-039555  (아드에스정)
28: K-039559  (프레가스타캡슐 75mg)
29: K-039560  (프레가스타캡슐 150mg)
30: K-039568  (프가바린캡슐 75mg)
31: K-039579  (자니엠정)
32: K-039588  (라이트레바정)
33: K-039610  (디오브이플러스정 80/12.5mg)
34: K-039633  (알러엔연질캡슐 10mg/PTP)
35: K-039659  (록스바정 5mg)
36: K-039703  (헤파텍트

# 확인 작업

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path(r"C:/project_sep/csv/antibiotics_c_code_normalized.csv")
df_csv = pd.read_csv(csv_path)

valid_kcodes = set(df_csv["C-Code"].astype(str).unique())
print("CSV 기준 K-code 수:", len(valid_kcodes))  # 343 나와야 함

CSV 기준 K-code 수: 343


In [4]:
import pandas as pd
from pathlib import Path

# 기존 samples.csv 불러오기
samples_path = Path(r"C:/project_sep/csv/samples.csv")
df = pd.read_csv(samples_path)

print("원본 samples.csv 크기:", df.shape)
print("원본 클래스 수:", df["k_code"].nunique())

# CSV 기준 343종 불러오기
csv_path = Path(r"C:/project_sep/csv/antibiotics_c_code_normalized.csv")
df_csv = pd.read_csv(csv_path)
valid_kcodes = set(df_csv["C-Code"].astype(str).unique())
print("CSV 기준 클래스 수:", len(valid_kcodes))  # 343

# 343종만 필터링
df = df[df["k_code"].isin(valid_kcodes)].reset_index(drop=True)

# class_id 다시 생성
df["class_id"] = df["k_code"].astype("category").cat.codes

print("필터링 후 데이터 크기:", df.shape)
print("필터링 후 클래스 수:", df["class_id"].nunique())

원본 samples.csv 크기: (160056, 5)
원본 클래스 수: 500
CSV 기준 클래스 수: 343
필터링 후 데이터 크기: (9612, 5)
필터링 후 클래스 수: 30


In [ ]:
print(df["k_code"].unique()[:20])

['K-043511' 'K-044312' 'K-044992' 'K-042035' 'K-043911' 'K-044927'
 'K-042736' 'K-044831' 'K-041092' 'K-044704' 'K-040733' 'K-040119'
 'K-041744' 'K-043508' 'K-044171' 'K-041676' 'K-041600' 'K-044872'
 'K-043376' 'K-039201']


In [6]:
print(df_csv["C-Code"].unique()[:20])

['K-000040' 'K-000066' 'K-000080' 'K-000105' 'K-000114' 'K-000230'
 'K-000257' 'K-000304' 'K-000400' 'K-000465' 'K-000608' 'K-000646'
 'K-000647' 'K-000653' 'K-000748' 'K-000774' 'K-000817' 'K-000818'
 'K-000954' 'K-001027']


In [7]:
import pandas as pd

# samples.csv 불러오기
df = pd.read_csv(r"C:/project_sep/csv/samples.csv")


# k_code를 6자리 zero-padding 포맷으로 변환
def normalize_kcode(k):
    try:
        num = int(k.split("-")[1])  # "K-043511" → 43511
        return f"K-{num:06d}"  # "K-043511" → "K-043511" (6자리 유지)
    except:
        return k


df["k_code_norm"] = df["k_code"].apply(normalize_kcode)

print("변환된 K-code 샘플:", df["k_code_norm"].unique()[:20])

변환된 K-code 샘플: ['K-040599' 'K-044528' 'K-042278' 'K-042274' 'K-042573' 'K-042584'
 'K-041273' 'K-042154' 'K-043065' 'K-039727' 'K-044334' 'K-040790'
 'K-045015' 'K-042854' 'K-042732' 'K-040100' 'K-041878' 'K-044754'
 'K-045255' 'K-040848']


In [8]:
# CSV 불러오기
df_csv = pd.read_csv(r"C:/project_sep/csv/antibiotics_c_code_normalized.csv")
valid_kcodes = set(df_csv["C-Code"].astype(str).unique())
print("CSV 기준 K-code 수:", len(valid_kcodes))  # 343

# 교집합 필터링
df_filtered = df[df["k_code_norm"].isin(valid_kcodes)].reset_index(drop=True)

# class_id 다시 생성
df_filtered["class_id"] = df_filtered["k_code_norm"].astype("category").cat.codes

print("필터링 후 샘플 수:", len(df_filtered))
print("필터링 후 클래스 수:", df_filtered["class_id"].nunique())

CSV 기준 K-code 수: 343
필터링 후 샘플 수: 9612
필터링 후 클래스 수: 30


In [9]:
# samples.csv에서 변환된 K-code set
sample_kcodes = set(df["k_code_norm"].unique())

# CSV의 343종 K-code set
csv_kcodes = set(df_csv["C-Code"].astype(str).unique())

# 교집합 / 차집합
common = sample_kcodes & csv_kcodes
only_in_samples = sample_kcodes - csv_kcodes
only_in_csv = csv_kcodes - sample_kcodes

print("공통된 K-code 개수:", len(common))
print("샘플에만 있는 K-code 개수:", len(only_in_samples))
print("CSV에만 있는 K-code 개수:", len(only_in_csv))

print("공통 예시:", list(common)[:20])
print("샘플에만 있는 예시:", list(only_in_samples)[:20])
print("CSV에만 있는 예시:", list(only_in_csv)[:20])

공통된 K-code 개수: 30
샘플에만 있는 K-code 개수: 470
CSV에만 있는 K-code 개수: 313
공통 예시: ['K-044312', 'K-044704', 'K-041344', 'K-043508', 'K-043911', 'K-042174', 'K-040119', 'K-042035', 'K-044992', 'K-041600', 'K-044948', 'K-043376', 'K-044872', 'K-040326', 'K-044831', 'K-042103', 'K-041092', 'K-044171', 'K-043511', 'K-040733']
샘플에만 있는 예시: ['K-042514', 'K-041102', 'K-045027', 'K-043999', 'K-041994', 'K-042107', 'K-041341', 'K-044437', 'K-045000', 'K-039759', 'K-042386', 'K-040248', 'K-039293', 'K-045014', 'K-041305', 'K-044592', 'K-040697', 'K-043334', 'K-039148', 'K-042404']
CSV에만 있는 예시: ['K-030912', 'K-005606', 'K-008451', 'K-014228', 'K-047816', 'K-003173', 'K-021771', 'K-017977', 'K-002691', 'K-008124', 'K-016866', 'K-019958', 'K-013103', 'K-031974', 'K-010731', 'K-006558', 'K-035833', 'K-031357', 'K-002896', 'K-030741']


In [10]:
import pandas as pd

df_samples = pd.read_csv("C:/project_sep/csv/samples.csv")
df_csv = pd.read_csv("C:/project_sep/csv/antibiotics_c_code_normalized.csv")

# 샘플 코드와 CSV 코드 확인
print("샘플 k_code 예시:", df_samples["k_code"].unique()[:20])
print("CSV C-Code 예시:", df_csv["C-Code"].unique()[:20])

샘플 k_code 예시: ['K-040599' 'K-044528' 'K-042278' 'K-042274' 'K-042573' 'K-042584'
 'K-041273' 'K-042154' 'K-043065' 'K-039727' 'K-044334' 'K-040790'
 'K-045015' 'K-042854' 'K-042732' 'K-040100' 'K-041878' 'K-044754'
 'K-045255' 'K-040848']
CSV C-Code 예시: ['K-000040' 'K-000066' 'K-000080' 'K-000105' 'K-000114' 'K-000230'
 'K-000257' 'K-000304' 'K-000400' 'K-000465' 'K-000608' 'K-000646'
 'K-000647' 'K-000653' 'K-000748' 'K-000774' 'K-000817' 'K-000818'
 'K-000954' 'K-001027']


In [1]:
import pandas as pd
from pathlib import Path

# 경로
samples_path = Path(r"C:/project_sep/csv/samples.csv")  # 500종 버전
csv_path = Path(r"C:/project_sep/csv/antibiotics_c_code.csv")     # 343종 버전

# 불러오기
df_samples = pd.read_csv(samples_path)
df_csv = pd.read_csv(csv_path)

print("samples.csv 클래스 수:", df_samples["k_code"].nunique())
print("CSV 기준 클래스 수:", df_csv["C-Code"].nunique())


samples.csv 클래스 수: 500
CSV 기준 클래스 수: 343


In [2]:
# samples.csv K-code 포맷 맞추기
def normalize_kcode(k):
    try:
        num = int(k.split("-")[1])
        return f"K-{num:06d}"
    except:
        return k

df_samples["k_code_norm"] = df_samples["k_code"].apply(normalize_kcode)

# CSV 기준
valid_kcodes = set(df_csv["C-Code"].astype(str).unique())


In [3]:
sample_kcodes = set(df_samples["k_code_norm"].unique())
common = sample_kcodes & valid_kcodes

print("samples.csv 안에 존재하는 CSV 클래스 수:", len(common))
print("예시 공통 K-code:", list(common)[:20])

samples.csv 안에 존재하는 CSV 클래스 수: 30
예시 공통 K-code: ['K-041092', 'K-039201', 'K-042736', 'K-043911', 'K-041744', 'K-044171', 'K-043468', 'K-041600', 'K-040119', 'K-039934', 'K-041344', 'K-044948', 'K-044992', 'K-043508', 'K-043511', 'K-044872', 'K-044927', 'K-041221', 'K-044831', 'K-042103']


In [ ]:
df_filtered = df_samples[df_samples["k_code_norm"].isin(common)].reset_index(drop=True)

# 숫자 라벨 다시 생성
df_filtered["class_id"] = df_filtered["k_code_norm"].astype("category").cat.codes

print("필터링 후 이미지 수:", len(df_filtered))
print("필터링 후 클래스 수:", df_filtered["class_id"].nunique())  # 343 나와야 정상


필터링 후 이미지 수: 9612
필터링 후 클래스 수: 30


In [5]:
print("교집합 개수:", len(common))         # 몇 종이 실제로 겹치나?
print("CSV에만 있는 클래스 수:", len(valid_kcodes - sample_kcodes))
print("samples에만 있는 클래스 수:", len(sample_kcodes - valid_kcodes))

print("CSV에만 있는 예시:", list(valid_kcodes - sample_kcodes)[:20])
print("samples에만 있는 예시:", list(sample_kcodes - valid_kcodes)[:20])


교집합 개수: 30
CSV에만 있는 클래스 수: 313
samples에만 있는 클래스 수: 470
CSV에만 있는 예시: ['K-009451', 'K-002236', 'K-002857', 'K-006557', 'K-005962', 'K-003173', 'K-003499', 'K-038727', 'K-031840', 'K-024297', 'K-035811', 'K-012080', 'K-020238', 'K-003361', 'K-038957', 'K-052476', 'K-038076', 'K-020641', 'K-027868', 'K-003700']
samples에만 있는 예시: ['K-043293', 'K-044127', 'K-042662', 'K-044077', 'K-044075', 'K-041549', 'K-044288', 'K-039893', 'K-043113', 'K-044126', 'K-043471', 'K-039338', 'K-041320', 'K-044619', 'K-044145', 'K-040128', 'K-041102', 'K-045269', 'K-045030', 'K-043887']


In [6]:
print("samples k_code 예시:", df_samples["k_code_norm"].unique()[:20])
print("CSV C-Code 예시:", list(valid_kcodes)[:20])


samples k_code 예시: ['K-040599' 'K-044528' 'K-042278' 'K-042274' 'K-042573' 'K-042584'
 'K-041273' 'K-042154' 'K-043065' 'K-039727' 'K-044334' 'K-040790'
 'K-045015' 'K-042854' 'K-042732' 'K-040100' 'K-041878' 'K-044754'
 'K-045255' 'K-040848']
CSV C-Code 예시: ['K-002236', 'K-009451', 'K-044171', 'K-002857', 'K-006557', 'K-005962', 'K-003173', 'K-003499', 'K-038727', 'K-031840', 'K-024297', 'K-042174', 'K-035811', 'K-012080', 'K-020238', 'K-003361', 'K-038957', 'K-052476', 'K-038076', 'K-020641']
